# AFM Regression Model


**This notebook provides a complete example on how to use a simulated AFM datase to train a regression model for post-processing.**

The first section simulates a dataset while the second section builds and trains a model. Finally, the user can test the model on their experimental data. The notebook provides two final **Output**; **1:** A simulated AFM dataset, **2:** A trained model ready to process experimental data.

## 1. Optional installations

Uncomment the lines if using Colab/Kaggle.

In [ ]:
#!pip3 install deeplay plotly ipywidgets ipympl

# from google.colab import output
# output.enable_custom_widget_manager()

## 2. Imports and configuration

Here we import necessary libraries and scripts.

In [ ]:
from __future__ import annotations

import glob
import importlib
import json
import logging
import os
import random
import sys
from copy import deepcopy
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Optional

import matplotlib
import matplotlib.animation
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import torch
import torch.nn as nn
from IPython.display import display
from matplotlib.pylab import maximum
from scipy.interpolate import interp1d
from sklearn.metrics import mean_squared_error, root_mean_squared_error
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

import deeplay as dl
import lightning as L
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from pytorch_lightning.loggers import CSVLogger

torch.set_num_threads(1)

We configure our simulator and model. The user must configure the experimental parameters, sample parameters and if desired the model parameters to adapt to their experimental setup.

1. **SimulationConfig**:
    1. The user must select the scan resolution that consists of pixel number $(n_x, n_y)$ and pixel size.
    2. Force-distance curve (FDC) parameters must be defined: $z~length$, $z~speed$, sampling frequency $f_s$ and $set-point~force$.
    3. Depending on the cantilever choice the tip shape and the desired thermal noise type must be selected.
    4. Curve mode must be selected between 'approach' and 'approach-retract'. This will depend on the used contact mechanics model.
3. **DatasetConfig**:
    1. Quantity of samples for training, validating and testing must be indicated.
    2. Topography types we want to use and the height range we are expecting.
    3. Range of stiffness for the substrate and the sample must be indicated. This is only an estimation and doesn't need to be precise.
5. **ModelConfig**:
    1. This contains the parameters of the model. The user does not need to adjust these values but it can b emodified if desired.

In [ ]:
SEED = 70
np.random.seed(SEED)
torch.manual_seed(SEED)

from afm_simulation import (
    AFMCantilever,
    AFMSimulator,
    DatasetConfig,
    HertzContact,
    ModelConfig,
    SampleGenerator,
    SimulationConfig,
    dataset_config,
    model_config,
    simulation_config,
)
from afm_plotting import plot_force_curve, plot_maps, plot_psd


## 3. Instantiate the simulator and generate one example sample

In [ ]:
# We use the simulator to generate data and visualise it with the functions
# defined above.
rng = np.random.default_rng(SEED)

example_substrate_stiffness_range_pa = (0.1e3, 5e3)
example_feature_stiffness_range_pa = (0.1e3, 5e3)
example_nx = 16
example_ny = 16

example_stiffness_min_pa = min(example_substrate_stiffness_range_pa[0],
                               example_feature_stiffness_range_pa[0])
example_stiffness_max_pa = max(example_substrate_stiffness_range_pa[1],
                               example_feature_stiffness_range_pa[1])

cantilever = AFMCantilever(k=0.178, f0=10e3, Q=100.0, T=300.0)
contact_model = HertzContact(R=1e-6, nu=0.5, alpha=18.0)
sample_generator = SampleGenerator(
    substrate_stiffness_range_pa=example_substrate_stiffness_range_pa,
    feature_stiffness_range_pa=example_feature_stiffness_range_pa,
    height_range_m=dataset_config.height_range_m
)
simulator = AFMSimulator(
    cantilever=cantilever,
    contact_model=contact_model,
    sample_generator=sample_generator,
    config=simulation_config,
)

print(f"Approach points = {simulator.n_points_approach}")
print(f"Setpoint force range = \
{min(simulation_config.setpoint_force_n_range) * 1e9:.2f} - \
{max(simulation_config.setpoint_force_n_range) * 1e9:.2f} nN")

example_topography_m, example_stiffness_pa = (
    sample_generator.create_topography(
        size=(example_nx, example_ny),
        pixel_size_m=simulation_config.pixel_size_m,
        topography_type="gaussian_features",
        rng=rng,
    )
)

example_output = simulator.simulate_force_map(
    stiffness_map_pa=example_stiffness_pa,
    topography_map_m=example_topography_m,
    include_thermal_noise=simulation_config.include_thermal_noise,
    rng=rng,
)

plot_maps(
    example_topography_m, 
    example_stiffness_pa, 
    example_output["contact_point_map_m"]
)

center_row = 0
center_col = 0
plot_force_curve(
    example_output["displacement_curves_m"][center_row, center_col],
    example_output["measured_force_curves_n"][center_row, center_col],
    retract_start_index= \
      int(example_output["retract_start_indices"][center_row, center_col]),
    valid_length=int(example_output["curve_lengths"][center_row, center_col]),
    title="Example measured force curve (centre pixel)",
)

reference_points = max(simulator.n_points_approach * 2, 64)
if simulation_config.noise_type == "rms":
    frequency_hz, spectral_density_m2_per_hz = cantilever.get_noise_psd(
        n_points=reference_points,
        sampling_frequency_hz=cantilever.f0*10,
        noise_type="theoretical",
        experimental_path=simulation_config.experimental_noise_path,
    )

else:
    frequency_hz, spectral_density_m2_per_hz = cantilever.get_noise_psd(
        n_points=reference_points,
        sampling_frequency_hz=cantilever.f0*10,
        noise_type=simulation_config.noise_type,
        experimental_path=simulation_config.experimental_noise_path,
    )
plot_psd(
    frequency_hz,
    spectral_density_m2_per_hz,
    cantilever,
    sampling_frequency=simulation_config.sampling_frequency_hz,
)


## 4. Curve Generation and Preprocessing

Generate the curves and interpolate the tip displament and measured force to fix length ...

In [ ]:
align_length = 1500

rng = np.random.default_rng(577)
ny = simulator.config.ny
nx = simulator.config.nx

stiffness_range_min = \
min(simulator.sample_generator.substrate_stiffness_range_pa[0], \
    simulator.sample_generator.feature_stiffness_range_pa[0])
stiffness_range_max = \
max(simulator.sample_generator.substrate_stiffness_range_pa[1], \
    simulator.sample_generator.feature_stiffness_range_pa[1])

def preprocess_synthetic_data(
    output: dict,
    stiffness_map_pa: np.ndarray,
    align_length: int,
    stiffness_range_min: float,
    stiffness_range_max: float,
    cut_nb: int = 10,
)-> tuple[np.ndarray, np.ndarray]:
    """Preprocess synthetic AFM data for regression.

    Parameters
    ----------
    output: dict
        Output from the AFMSimulator containing force curves and metadata.
    stiffness_map_pa: np.ndarray
        Stiffness map in pascals.
    align_length: int
        Length to which all curves will be aligned (via interpolation).
    stiffness_range_min: float
        Minimum stiffness value for normalization.
    stiffness_range_max: float
        Maximum stiffness value for normalization.

    Returns
    -------
    tuple[np.ndarray, np.ndarray]
        Tuple of (data_x, data_y) where data_x contains the aligned force
        curves and data_y contains the corresponding contact points and
        stiffness values.

    """

    ny = output["curve_lengths"].shape[0]
    nx = output["curve_lengths"].shape[1]

    n_total = ny * nx * cut_nb
    data_x = np.empty((n_total, align_length * 2), dtype=np.float32) 
    data_y = np.empty((n_total, 2), dtype=np.float32)
    idx = 0

    for r in range(ny):
        for c in range(nx):
            disp_all = output["displacement_curves_m"][r, c]
            disp_all = np.asarray(disp_all, dtype=np.float64)

            meas_all = output["measured_force_curves_n"][r, c]
            meas_all = np.asarray(meas_all, dtype=np.float64)
            stif = float(stiffness_map_pa[r, c])
            cp_og = float(output["contact_point_map_m"][r, c])

            for j in range(cut_nb):
                cp = deepcopy(cp_og)
                stif = deepcopy(stif)

                disp = disp_all.copy()
                meas = meas_all.copy()

                cp = disp.max() - cp
                disp = disp.max() - disp
                cp = cp/disp.max()

                cut1 = int(random.random()*0.5*cp*disp.size)
                cut2 = int((1-(1-cp)*random.random()*0.9)*disp.size)

                if cut1 >= cut2 or j==0:
                    cut1 = 0
                    cut2 = disp.size

                cp = cp*disp.max()

                disp = disp[::-1][cut1:cut2]
                meas = meas[::-1][cut1:cut2]

                x_old = np.linspace(0, 1, disp.size)
                x_new = np.linspace(0, 1, align_length)

                disp = np.interp(x_new, x_old, disp)
                meas = np.interp(x_new, x_old, meas)

                cp = cp - disp.min()
                disp = disp - disp.min()
                cp = cp/disp.max()

                disp = disp[::-1]
                meas = meas[::-1]

                data_x[idx] = np.hstack([disp*1e6,meas*1e9])
                data_y[idx] = [cp, stif]
                idx += 1


    data_y[:,1] = (data_y[:,1] - stiffness_range_min)/\
     (stiffness_range_max - stiffness_range_min)

    return data_x, data_y

cut_nb = 10
n_total = dataset_config.n_samples * nx * ny * cut_nb
data_x_r = np.empty((n_total, align_length * 2), dtype=np.float32)
data_y_r = np.empty((n_total, 2), dtype=np.float32)
idx = 0
cp_og = []
for i in range(dataset_config.n_samples):
    topography_type = rng.choice(dataset_config.topography_types)
    topo_m, stiff_pa = sample_generator.create_topography(
        size=(ny, nx),
        pixel_size_m=simulator.config.pixel_size_m,
        topography_type=topography_type,
        rng=rng,
    )
    out = simulator.simulate_force_map(
        stiffness_map_pa=stiff_pa,
        topography_map_m=topo_m,
        include_thermal_noise=simulator.config.include_thermal_noise,
        noise_type=simulator.config.noise_type,
        experimental_path=simulator.config.experimental_noise_path,
        rng=rng,
    )

    data_x_i, data_y_i = preprocess_synthetic_data(
        out,
        stiff_pa,
        align_length,
        stiffness_range_min,
        stiffness_range_max,
        cut_nb=cut_nb,
    )

    data_x_r[idx : idx + data_x_i.shape[0]] = data_x_i
    data_y_r[idx : idx + data_y_i.shape[0]] = data_y_i
    idx += data_x_i.shape[0]
    cp_og.append(out["contact_point_map_m"].ravel())

    del out, topo_m, stiff_pa


print(f"Final dataset shape: data_x={data_x_r.shape}, data_y={data_y_r.shape}") 

In [ ]:
plt.hist(data_y_r[:,0], bins=50)
plt.xlabel("Contact point (normalized)")
plt.ylabel("Frequency")
plt.title("Contact point vs Stiffness in the synthetic dataset")
plt.grid(alpha=0.3)
plt.show()

cp_og = np.concatenate(cp_og)

plt.hist(cp_og, bins=50)
plt.xlabel("Contact point (original)")
plt.ylabel("Frequency")
plt.title("Original contact point distribution in the synthetic dataset")
plt.grid(alpha=0.3)
plt.show()

plt.hist(data_y_r[:,1], bins=50)
plt.xlabel("Stiffness (normalized)")
plt.ylabel("Frequency")
plt.title("Stiffness distribution in the synthetic dataset")
plt.grid(alpha=0.3) 
plt.show()



... and create the dataloader for tranining.

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    data_x_r, data_y_r,
    test_size=0.3,
    random_state=577
)

train_dataset = TensorDataset(
    torch.tensor(x_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.float32)
)

val_dataset = TensorDataset(
    torch.tensor(x_test, dtype=torch.float32),
    torch.tensor(y_test, dtype=torch.float32)
)

train_loader = DataLoader(
    train_dataset, 
    batch_size=model_config.batch_size, 
    shuffle=True
    )
val_loader = DataLoader(
    val_dataset, 
    batch_size=model_config.batch_size, 
    shuffle=True
    )

## 5. MLP Training

We instantiate a dense layers neural network containing two heads, one for contact point estimation and one for stiffness estimation.

In [ ]:
class MLP(L.LightningModule):
    """AFMCantilever regression MLP model.

    Parameters
    ----------
    lr: float
        Learning rate for the optimizer.
    
    Methods
    -------
    `forward(x) -> torch.Tensor`
        Forward pass of the MLP model.
    `training_step(batch, batch_idx) -> torch.Tensor`
        Training step for the MLP model.
    `validation_step(batch, batch_idx) -> torch.Tensor`
        Validation step for the MLP model.
    `test_step(batch, batch_idx) -> torch.Tensor`
        Test step for the MLP model.
    `configure_optimizers() -> torch.optim.Optimizer`
        Configure the optimizer for training.
    
    """
        
    def __init__(
        self: MLP, 
        lr: float = 1e-4
    ):
        """Initializes the MLP model for regression.
        
        Parameters
        ----------
        lr: float
            Learning rate for the optimizer.
        
        Returns
        -------
        None

        """
        super().__init__()
        self.save_hyperparameters()

        self.encoder = nn.Sequential(
            nn.Linear(3000, 1024),
            nn.LeakyReLU(),
            nn.Linear(1024, 512),
            nn.LeakyReLU(),
        )

        self.cp_head = nn.Sequential(
            nn.Linear(512, 256),
            nn.LeakyReLU(),
            nn.Linear(256, 1),
        )

        self.sf_head = nn.Sequential(
            nn.Linear(513, 256),
            nn.LeakyReLU(),
            nn.Linear(256, 1),
        )

        self.loss_fn = nn.MSELoss()

    def forward(
        self: MLP, 
        x: torch.Tensor
    ) -> torch.Tensor:
        """Forward pass of the MLP model.

        Parameters
        ----------
        x: torch.Tensor
            Input tensor containing the aligned force curves, 
            shape (batch_size, 3000).
            
        Returns
        -------
        torch.Tensor
            Output tensor containing the predicted contact points and stiffness 
            values, shape (batch_size, 2).

        """

        feat = self.encoder(x)
        cp = self.cp_head(feat)

        stiffness_input = torch.cat([feat, cp], dim=1)
        stiffness = self.sf_head(stiffness_input)

        #stiffness = self.sf_head(feat) #non joint

        # Final output: [cp, stiffness]
        y = torch.cat([cp, stiffness], dim=1)
        return y
        
    def training_step(
        self: MLP, 
        batch: tuple[torch.Tensor, torch.Tensor],
        batch_idx: int
    ) -> torch.Tensor:
        """Training step for the MLP model.

        Parameters
        ----------
        batch: tuple[torch.Tensor, torch.Tensor]
            A tuple containing the input tensor and the target tensor.
        batch_idx: int
            Index of the batch.

        Returns
        -------
        torch.Tensor
            The computed loss for the batch.
            
        """

        x, y = batch
        y_hat = self(x)
        loss = self.loss_fn(y_hat, y)
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(
        self: MLP, 
        batch: tuple[torch.Tensor, torch.Tensor],
        batch_idx: int
    ) -> torch.Tensor:
        """Validation step for the MLP model.

        Parameters
        ----------
        batch: tuple[torch.Tensor, torch.Tensor]
            A tuple containing the input tensor and the target tensor.
        batch_idx: int
            Index of the batch.

        Returns
        -------
        torch.Tensor
            The computed loss for the batch.
        """

        x, y = batch
        y_hat = self(x)
        loss = self.loss_fn(y_hat, y)
        self.log("val_loss", loss, prog_bar=True)
        return loss

    def test_step(
        self: MLP, 
        batch: tuple[torch.Tensor, torch.Tensor],
        batch_idx: int
    ) -> torch.Tensor:
        """Test step for the MLP model.

        Parameters
        ----------
        batch: tuple[torch.Tensor, torch.Tensor]
            A tuple containing the input tensor and the target tensor.
        batch_idx: int
            Index of the batch.

        Returns
        -------
        torch.Tensor
            The computed loss for the batch.
        """
        
        x, y = batch
        y_hat = self(x)
        loss = self.loss_fn(y_hat, y)
        self.log("test_loss", loss, prog_bar=True)
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)
    
regressor = MLP()

print(regressor)

We define `RunSaver` utility class that organises each experiment into a `runs/<run_name>/` directory, persisting the configuration, per-epoch metrics, model checkpoints, and final test results as JSON files. Also provides `build_run_config`, which flattens `SimulationConfig`, `DatasetConfig`, and `ModelConfig` into a single JSON-serialisable dictionary for logging.

In [ ]:
import json
import logging
from datetime import datetime
from pathlib import Path


class RunSaver:
    """
    Creates runs/<run_name>/ and saves:
      config.json    — all hyperparameters + ablation flags + data conditions
      metrics.json   — per-epoch train/val loss (parsed from Lightning CSV log)
      results.json   — test metrics → feeds the ablation table
      logs/training.log — human-readable log
    """

    def __init__(
        self,
        run_name: str,
        config: dict,
        model_structure: str = "MLP",
        base_dir: str = "runs",
    ) -> None:
        self.run_name        = run_name
        self.config          = config
        self.model_structure = model_structure
        self.run_dir         = Path(base_dir) / run_name
        self.run_dir.mkdir(parents=True, exist_ok=True)

        self.best_val_loss = float("inf")
        self.best_epoch    = 0

        self._init_logger()
        self._save_json(config, "config.json")
        self.logger.info(f"Run: {run_name}")

    def _init_logger(self) -> None:
        log_dir = self.run_dir / "logs"
        log_dir.mkdir(exist_ok=True)
        logger = logging.getLogger(self.run_name)
        logger.setLevel(logging.INFO)
        if not logger.handlers:
            fh = logging.FileHandler(log_dir / "training.log")
            sh = logging.StreamHandler()
            fmt = logging.Formatter("%(asctime)s  %(message)s", "%H:%M:%S")
            fh.setFormatter(fmt)
            sh.setFormatter(fmt)
            logger.addHandler(fh)
            logger.addHandler(sh)
        self.logger = logger

    def _save_json(self, data: dict, filename: str) -> None:
        def _convert(obj):
            if isinstance(obj, np.integer):  return int(obj)
            if isinstance(obj, np.floating): return float(obj)
            if isinstance(obj, np.ndarray):  return obj.tolist()
            if isinstance(obj, tuple):       return list(obj)
            return obj
        with open(self.run_dir / filename, "w") as f:
            json.dump(
                {k: _convert(v) for k, v in data.items()}
                if isinstance(data, dict) else data,
                f, indent=2,
            )

    def load_history_from_csv(self, csv_path: str) -> dict:
        """Parse a Lightning CSVLogger metrics.csv into a history dict."""
        import pandas as pd
        metrics  = pd.read_csv(csv_path)
        # Lightning may log train_loss as "train_loss", "train_loss_step", or
        # "train_loss_epoch" depending on on_step/on_epoch settings.
        train_col = next(
            (c for c in ["train_loss", "train_loss_epoch", "train_loss_step"]
            if c in metrics.columns),
            None,
        )
        val_col = next(
            (c for c in ["val_loss", "val_loss_epoch"]
            if c in metrics.columns),
            None,
        )

        if train_col is None or val_col is None:
            print(f"Available columns: {list(metrics.columns)}")
            print("Could not find train_loss or val_loss columns.")
        else:
            # val_loss is logged once per epoch; train_loss logged every N
            # steps — average per epoch so both share the same x-axis
            train_metrics = (
                metrics[metrics[train_col].notna()]
                .groupby("epoch", as_index=False)[train_col]
                .mean()
            )
            val_metrics = metrics[metrics[val_col].notna()][["epoch", val_col]]
        train_df = train_metrics if 'train_metrics' in locals() else \
            pd.DataFrame(columns=['epoch', 'train_loss'])
        val_df = val_metrics if 'val_metrics' in locals() else\
            pd.DataFrame(columns=['epoch', 'val_loss'])
        return {
            "train_loss": train_df["train_loss"].tolist(),
            "val_loss":   val_df["val_loss"].tolist(),
        }

    def save_history(self, history: dict) -> None:
        """Persist metrics.json and update best_epoch / best_val_loss."""
        self._save_json(history, "metrics.json")
        val_losses = history.get("val_loss", [])
        if val_losses:
            self.best_epoch    = int(np.argmin(val_losses)) + 1
            self.best_val_loss = float(np.min(val_losses))

    def save_checkpoint(self, model: nn.Module, tag: str) -> None:
        """Save model state dict to model_<tag>.pth."""
        path = self.run_dir / f"model_{tag}.pth"
        torch.save(model.state_dict(), path)
        self.logger.info(f"Checkpoint saved → {path}")

    def save_results(
        self,
        test_mse_cp: float,
        test_rmse_cp: float,
        test_mse_stiffness: float,
        test_rmse_stiffness: float,
        num_params: int,
    ) -> dict:
        """Build results dict, write results.json, and return the dict."""
        results = {
            # identifier
            "run_name":              self.run_name,
            # model structure (ablation axis)
            "model_structure":       self.model_structure,
            # model / training (ablation axes)
            "learning_rate":         self.config.get("learning_rate"),
            "batch_size":            self.config.get("batch_size"),
            "max_epochs":            self.config.get("max_epochs"),
            "predict_contact_point": self.config.get("predict_contact_point"),
            # simulation conditions (ablation axes)
            "tip_shape":             self.config.get("tip_shape"),
            "include_thermal_noise": self.config.get("include_thermal_noise"),
            "noise_type":            self.config.get("noise_type"),
            "curve_mode":            self.config.get("curve_mode"),
            "multi_modal":           self.config.get("multi_modal"),
            "range_multi_modal":     self.config.get("range_multi_modal"),
            # data conditions
            "n_samples":             self.config.get("n_samples"),
            "topography_types":      self.config.get("topography_types"),
            # performance
            "test_mse_cp":           round(test_mse_cp,          6),
            "test_rmse_cp":          round(test_rmse_cp,         6),
            "test_mse_stiffness":    round(test_mse_stiffness,   6),
            "test_rmse_stiffness":   round(test_rmse_stiffness,  6),
            "best_val_loss":         round(self.best_val_loss,    6),
            "best_epoch":            self.best_epoch,
            "num_params":            num_params,
            # reproducibility
            "seed":                  self.config.get("seed"),
            "timestamp":             datetime.now().isoformat(),
        }
        self._save_json(results, "results.json")
        self.logger.info(
            f"Results saved — "
            f"test_rmse_cp={test_rmse_cp:.6f}  "
            f"test_rmse_stiffness={test_rmse_stiffness:.6f}"
        )
        return results

def build_run_config(
    sim_cfg: SimulationConfig,
    data_cfg: DatasetConfig,
    model_cfg: ModelConfig,
    seed: int = SEED,
) -> dict:
    """Flatten the three config dataclasses into one JSON-serialisable dict."""
    return {
        # SimulationConfig — ablation axes
        "tip_shape":                     sim_cfg.tip_shape,
        "include_thermal_noise":         sim_cfg.include_thermal_noise,
        "noise_type":                    sim_cfg.noise_type,
        "curve_mode":                    sim_cfg.curve_mode,
        "multi_modal":                   sim_cfg.multi_modal,
        # SimulationConfig — experimental conditions
        "z_length_m":                    sim_cfg.z_length_m,
        "z_speed_um_s":                  sim_cfg.z_speed_um_s,
        "sampling_frequency_hz":         sim_cfg.sampling_frequency_hz,
        "setpoint_force_n_range":        list(sim_cfg.setpoint_force_n_range),
        # DatasetConfig
        "n_samples":                     data_cfg.n_samples,
        "topography_types":              list(data_cfg.topography_types),
        "substrate_stiffness_range_pa":  \
            list(data_cfg.substrate_stiffness_range_pa),
        "feature_stiffness_range_pa":    \
            list(data_cfg.feature_stiffness_range_pa),
        "height_range_m":                list(data_cfg.height_range_m),
        # ModelConfig
        "batch_size":                    model_cfg.batch_size,
        "learning_rate":                 model_cfg.learning_rate,
        "max_epochs":                    model_cfg.max_epochs,
        "predict_contact_point":         model_cfg.predict_contact_point,
        "cp_loss_weight":                model_cfg.cp_loss_weight,
        # Reproducibility
        "seed":                          seed,
    }


print("RunSaver and build_run_config defined.")


... compile it ...

In [ ]:
# Generate run_name upfront so all artefacts land in one directory
run_name = f"{datetime.now().strftime('%Y%m%d_%H%M%S')}"
run_dir  = Path("runs") / run_name
run_dir.mkdir(parents=True, exist_ok=True)

# # Describe the model architecture (becomes a column in the ablation table)
model_structure = f"Full Model + randomn noise and bimodal stiffness"

# Callbacks, logger, and trainer — all outputs routed into runs/<run_name>/
checkpoint_callback = ModelCheckpoint(
    dirpath=str(run_dir / "checkpoints"),
    monitor="val_loss",
    mode="min",
    save_top_k=3,
)

early_stop_callback = EarlyStopping(
    monitor="val_loss",
    patience=40,
    mode="min",
    verbose=True,
)

csv_logger = CSVLogger(str(run_dir), name="logs", version=0)

trainer = dl.Trainer(
    max_epochs=model_config.max_epochs,
    accelerator=model_config.accelerator,
    callbacks=[checkpoint_callback],
    logger=csv_logger,
    log_every_n_steps=5,
    enable_progress_bar=True,
) 

... and print the summary of the compiled model.

In [ ]:
trainer.fit(regressor, train_loader, val_loader)


We read the Lightning `CSVLogger` metrics CSV file, groups train loss per epoch, and plots training and validation loss curves side by side to visualise model convergence.

In [ ]:

# Find the latest version log directory
metrics_path = str(Path(csv_logger.log_dir) / "metrics.csv")

if Path(metrics_path).exists():
    metrics = pd.read_csv(metrics_path)

    # Lightning may log train_loss as "train_loss", "train_loss_step", or
    # "train_loss_epoch" depending on on_step/on_epoch settings.
    train_col = next(
        (c for c in ["train_loss", "train_loss_epoch", "train_loss_step"]
         if c in metrics.columns),
        None,
    )
    val_col = next(
        (c for c in ["val_loss", "val_loss_epoch"]
         if c in metrics.columns),
        None,
    )

    if train_col is None or val_col is None:
        print(f"Available columns: {list(metrics.columns)}")
        print("Could not find train_loss or val_loss columns.")
    else:
        # val_loss is logged once per epoch; train_loss may be logged every N
        # steps — average per epoch so both share the same x-axis
        train_metrics = (
            metrics[metrics[train_col].notna()]
            .groupby("epoch", as_index=False)[train_col]
            .mean()
        )
        val_metrics = metrics[metrics[val_col].notna()][["epoch", val_col]]

        fig, ax = plt.subplots(figsize=(9, 4))
        ax.plot(
            train_metrics["epoch"],
            train_metrics[train_col],
            label="Train loss",
            alpha=0.8
            )
        ax.plot(
            val_metrics["epoch"],
            val_metrics[val_col],
            label="Val loss",
            alpha=0.8
            )
        ax.set_xlabel("Epoch")
        ax.set_ylabel("MSE Loss")
        ax.set_title("Training and Validation Loss")
        ax.legend()
        ax.grid(alpha=0.3)
        plt.tight_layout()
        plt.show()
        print(f"Loaded from: {metrics_path}")
else:
    print("No metrics.csv found. Run training first.")


## 6. Evaluate on the Generated Map

Load the best model from the callback function.

In [ ]:
LOAD_PRETRAINED_MODEL = False

if LOAD_PRETRAINED_MODEL:
    pretraiend_model_path = "runs/20260818_164457/checkpoints/\
        epoch=29-step=10500.ckpt"

    print(f"Loading the model at {pretraiend_model_path}")

    best_regressor = MLP.load_from_checkpoint(
        pretraiend_model_path
    )

else:
    print(f"The best model is saved at {checkpoint_callback.best_model_path}")

    best_regressor = MLP.load_from_checkpoint(
        checkpoint_callback.best_model_path
    )

Plot the map-level example again ...

In [ ]:
plot_maps(
    example_topography_m, 
    example_stiffness_pa, 
    example_output["contact_point_map_m"]
)

... process them for model inference and predict the contact points and stiffness ...

In [ ]:
data_input, data_target = preprocess_synthetic_data(
    example_output,
    example_stiffness_pa,
    align_length,
    example_stiffness_min_pa,
    example_stiffness_max_pa,
    cut_nb=1,
)

test_dataset = TensorDataset(
    torch.tensor(data_input, dtype=torch.float32),
    torch.tensor(data_target, dtype=torch.float32)
)

test_loader = DataLoader(test_dataset, batch_size=nx*ny, shuffle=False)

Calculate the metrics in generated map ...

In [ ]:
_ = trainer.test(best_regressor, test_loader)

We define `pass_through_model` function, which runs a full batched forward pass, collects raw inputs, ground-truth targets, and predictions, then denormalises both outputs back to physical units (µm for contact point, Pa for stiffness). Than we apply the fucntion to the simulated example map.

In [ ]:
device = next(best_regressor.parameters()).device

def pass_through_model(
    model: nn.Module,
    dataloader: DataLoader,
    device: torch.device,
    stiffness_range_min: float,
    stiffness_range_max: float,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Pass data through the model and collect predictions.

    Flat (batch, 2*align_length) inputs are handled — AFMStiffnessRegressor
    reshapes internally before the forward pass.
    Columns in returned arrays follow the [cp, stiffness] convention.
    """

    all_y = []
    all_y_hat = []
    all_x = []
    with torch.no_grad():
        for batch in dataloader:
            x, y = batch
            x = x.to(device)
            y = y.to(device)

            y_hat = model(x)
            all_x.append(x)
            all_y.append(y)
            all_y_hat.append(y_hat)

    all_x = torch.cat(all_x, dim=0)
    all_y = torch.cat(all_y, dim=0)
    all_y_hat = torch.cat(all_y_hat, dim=0)

    all_x = all_x.cpu().numpy()
    all_y = all_y.cpu().numpy()
    all_y_hat = all_y_hat.cpu().numpy()

    # Denormalize: columns are [cp, stiffness]
    all_y_hat[:, 1] = (all_y_hat[:, 1]
                       * (stiffness_range_max - stiffness_range_min)
                       + stiffness_range_min)
    all_y_hat[:, 0] = all_y_hat[:, 0] * all_x[:, 0:align_length].max(axis=1)
    all_y[:, 0] = all_y[:, 0] * all_x[:, 0:align_length].max(axis=1)
    all_y[:, 1] = (all_y[:, 1]
                   * (stiffness_range_max - stiffness_range_min)
                   + stiffness_range_min)

    return all_x, all_y, all_y_hat

all_x_example, all_y_example, all_y_hat_example = pass_through_model(
    best_regressor,
    test_loader,
    device,
    example_stiffness_min_pa,
    example_stiffness_max_pa,
)


We plot parity scatter plots comparing the true vs predicted contact point (left) and stiffness (right) on the simulated example map, with a dashed diagonal as the perfect-prediction reference.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(all_y_example[:,0],
                all_y_hat_example[:,0],
                alpha=0.4,
                s=12)
lim = [0, max(all_y_hat_example[:,0].max(), all_y_example[:,0].max())]
axes[0].plot(lim, lim, "k--", lw=1.5, label="Perfect")
axes[0].set_xlim(lim)
axes[0].set_ylim(lim)
axes[0].set_aspect("equal", adjustable="box")
axes[0].set_xlabel("MSE Fit [um]", fontsize=12)
axes[0].set_ylabel("ML Regression Fit [um]", fontsize=12)
axes[0].set_title("Contact Point", fontsize=15)

axes[1].scatter(all_y_example[:,1],
                all_y_hat_example[:,1],
                alpha=0.4,
                s=12,
                color="tab:orange",)
lim = [0,5000]
axes[1].plot(lim, lim, "k--", lw=1.5, label="Perfect")
# axes[1].set_xlim(lim)
# axes[1].set_ylim(lim)
axes[1].set_aspect("equal", adjustable="box")
axes[1].set_xlabel("MSE Fit [um]", fontsize=12)
axes[1].set_ylabel("ML Regression Fit [um]", fontsize=12)
axes[1].set_title("Young's Modulus", fontsize=15)

plt.tight_layout()
plt.show()

Clickable stiffnes map-viewer to visualize the output.

In [ ]:
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    output = None

try:
    import anywidget
    import ipywidgets as widgets
    import plotly.graph_objects as go
    PLOTLY_AVAILABLE = True
    print("Plotly FigureWidget is available.")
except Exception as error:
    widgets = None
    go = None
    PLOTLY_AVAILABLE = False
    print("Interactive Plotly viewer is unavailable in this environment.")
    print("Install these packages locally and restart the kernel:")
    print("pip install plotly anywidget ipywidgets")
    print(f"Import error: {error}")

sys.path.append(os.path.abspath("../scripts"))
import plotting
plotting = importlib.reload(plotting)

predicted_stiffness_map_pa = np.zeros_like(example_stiffness_pa)
predicted_contact_point_map_m = np.zeros_like(example_stiffness_pa)
approach_force = np.empty((example_nx, example_ny), dtype=object)
approach_force_gt = np.empty((example_nx, example_ny), dtype=object)


for r in range(example_nx):
    for c in range(example_ny):
        idx = r * example_ny + c
        cp = all_y_hat_example[idx, 0]
        stiffness = all_y_hat_example[idx, 1]
        disp = example_output["displacement_curves_m"][r, c]
        cp = cp/1e6
        cp = disp.max() - cp
        indent = np.maximum(disp - cp, 0.0)

        approach_force[r, c] = contact_model.force(
            cantilever_shape=simulation_config.tip_shape,
            indentation_m=indent,
            stiffness_pa=stiffness,
        )
        
        del indent
        predicted_contact_point_map_m[r, c] = cp
        predicted_stiffness_map_pa[r, c] = stiffness


plotting.display_clickable_stiffness_map(
    test_collection=example_output,
    example_stiffness_map_pa = example_stiffness_pa,
    predicted_stiffness_map_pa=predicted_stiffness_map_pa,
    approach_force=approach_force,
    predicted_contact_point_map_m=predicted_contact_point_map_m,
    contact_model=HertzContact,
    simulation_config=SimulationConfig,
    abstract_figure=True,
    save_dir='results/fitting_examples'
    )

## 7. Evaluate on the Real Data

Download and load the 3T3 dataset. Preprocess the data to prepare for the model. Pass the dataset through the model to obtain the predictions.

In [ ]:
cut_percentage = 0

if not os.path.exists("../data/3t3_cell_dataset"):
    os.system("git clone https://github.com/DeepTrackAI/3t3_cell_dataset")


# Load the approach curves
approach_path = Path.cwd() / "../data/3t3_cell_dataset" / "approach"
approach_files = sorted(approach_path.glob("*.npy"), key=lambda f: int(f.stem))
approach_curves = [np.load(f) for f in approach_files]

# Load the retraction curves
retract_path = Path.cwd() / "../data/3t3_cell_dataset" / "retraction"
retract_files = sorted(retract_path.glob("*.npy"), key=lambda f: int(f.stem))
retract_curves = [np.load(f) for f in retract_files]

# Load the labels
label_file = Path.cwd() / "../data/3t3_cell_dataset" / "label.npy"
labels = np.load(label_file)

# Load the contact points (µm) and young modulus (Pa)
contactpoint_file = Path.cwd() / "../data/3t3_cell_dataset" / "contact_point.npy"
contactpoints = np.load(contactpoint_file)          # µm

young_modulus_file = Path.cwd() / "../data/3t3_cell_dataset" / "young_modulus.npy"
young_modulus = np.load(young_modulus_file)          # Pa

# Calculate the length of each curve
approach_curve_lengths = [curve.shape[0] for curve in approach_curves]
retract_curve_lengths  = [curve.shape[0] for curve in retract_curves]
min_approach_curve_length = min(approach_curve_lengths)
min_retract_curve_length  = min(retract_curve_lengths)

print(
    f"Minimum approach curve length = {min_approach_curve_length}\n"
    f"Minimum retraction curve length = {min_retract_curve_length}\n"
)

def resample_to_same_length(
    data_list: list[np.ndarray],
    target_len: int
) -> list[np.ndarray]:
    """Resample a list of 1D arrays to the same length using linear
    interpolation.

    Parameters
    ----------
    data_list: list[np.ndarray]
        List of 1D arrays to be resampled.
    target_len: int
        Target length for the resampled arrays.

    Returns
    -------
    list[np.ndarray]
        List of resampled 1D arrays.

    """

    resampled = []
    for data in data_list:
        data = np.asarray(data)
        x_old = np.linspace(0, 1, len(data))
        x_new = np.linspace(0, 1, target_len)
        resampled.append(np.interp(x_new, x_old, data))
    return resampled

data_exp_x1 = []
data_exp_x2 = []
contactpoints_exp = []

for i, curve in enumerate(approach_curves):
    contactpoints_exp.append(contactpoints[i] - curve[:, 0].min())  \
        # contact point (µm)
    disp = curve[:, 0] - curve[:, 0].min()  # displacement (µm), decreasing
    force = curve[:, 1]                   # force (nN)
    disp = disp[0::]
    force = force[0::]
    data_exp_x1.append(disp)   # displacement (µm), decreasing
    data_exp_x2.append(force)   # force (nN)

teste1 = np.array(resample_to_same_length(data_exp_x1, align_length))
teste2 = np.array(resample_to_same_length(data_exp_x2, align_length))

denom = teste1.max(axis=1)
denom = np.where(denom == 0, np.nan, denom)  # avoid division by zero
contactpoints_exp = np.array(contactpoints_exp) / denom
#contactpoints_exp = np.array(contactpoints) / denom
data_xe_r = np.hstack([teste1, teste2])
data_ye_r = np.vstack([contactpoints_exp, young_modulus]).T
data_ye_r[:,1] = (data_ye_r[:,1] - stiffness_range_min)/\
 (stiffness_range_max - stiffness_range_min)

real_dataset = TensorDataset(
    torch.tensor(data_xe_r, dtype=torch.float32),
    torch.tensor(data_ye_r, dtype=torch.float32)
)

exp_loader = DataLoader(real_dataset, batch_size=data_xe_r.shape[0])

all_x_exp, all_y_exp, all_y_hat_exp = pass_through_model(
    best_regressor,
    exp_loader,
    device,
    stiffness_range_min,
    stiffness_range_max
)

cut_percentages = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

for iii in range(len(cut_percentages)):

    cut_percentage = cut_percentages[iii]

    data_exp_x1 = []
    data_exp_x2 = []
    contactpoints_exp_cut = []

    for ii in range(len(approach_curves)):
        i = approach_curves[ii]
        contactpoints_exp_cut.append(contactpoints[ii] -i[:, 0].min())  \
            # contact point (µm)
        disp = i[:, 0] - i[:, 0].min()  # displacement (µm), decreasing
        force = i[:, 1]   # force (nN)

        disp_range = disp.max()  # = i[:, 0].max() - i[:, 0].min()

        if all_y_hat_exp[ii, 0] < 0:
            disp = disp[int(i.shape[0]*cut_percentage)::]
            data_exp_x1.append(disp)
            force = force[int(i.shape[0]*cut_percentage)::]
            data_exp_x2.append(force)
        else:
            # all_y_hat_exp[:,0] is denormalized (µm); normalize back to [0,1]
            # by dividing by the curve's displacement range before using as a
            # fraction in the cut index, to avoid negative slice indices that
            # wrap to the end of the array and produce all-zero sub-arrays.
            if disp_range > 0:
                cp_frac = float(np.clip(all_y_hat_exp[ii, 0] / \
                                        disp_range, 0.0, 1.0))
            else:
                cp_frac = 0.0
            cut_start = int(i.shape[0] * (1 - cp_frac) * cut_percentage)
            disp = disp[cut_start::]
            force = force[cut_start::]
            data_exp_x1.append(disp)
            data_exp_x2.append(force)
    
    teste1 = resample_to_same_length(data_exp_x1, 1500)
    teste2 = resample_to_same_length(data_exp_x2, 1500)

    teste1 = np.array(teste1)
    teste2 = np.array(teste2)
    contactpoints_exp_cut = np.array(contactpoints_exp_cut)

    denom = teste1.max(axis=1)
    denom = np.where(denom == 0, np.nan, denom)  # avoid division by zero
    contactpoints_exp_cut = contactpoints_exp_cut - teste1.min(axis=1)  
    # ensure contact point is relative to curve start
    contactpoints_exp_cut = contactpoints_exp_cut / denom
    teste1 = teste1 - teste1.min(axis=1).reshape(-1,1)
    data_xe_r = np.hstack([teste1, teste2])
    data_ye_r = np.vstack([contactpoints_exp_cut, young_modulus]).T
    data_ye_r[:,1] = (data_ye_r[:,1] - stiffness_range_min)/\
     (stiffness_range_max - stiffness_range_min)

    if iii == 0:
        data_xe_r_list = data_xe_r.copy()
        data_ye_r_list = data_ye_r.copy()
    else:
        data_xe_r_list = np.vstack([data_xe_r_list, data_xe_r.copy()])
        data_ye_r_list = np.vstack([data_ye_r_list, data_ye_r.copy()])

real_dataset_cut = TensorDataset(
    torch.tensor(data_xe_r_list, dtype=torch.float32),
    torch.tensor(data_ye_r_list, dtype=torch.float32)
)

exp_loader_cut = DataLoader(
    real_dataset_cut,
    batch_size=data_xe_r_list.shape[0]
    )

all_x_exp_cut, all_y_exp_cut, all_y_hat_exp_cut = pass_through_model(
    best_regressor,
    exp_loader_cut,
    device,
    stiffness_range_min,
    stiffness_range_max
)

all_y_exp_cut = all_y_exp_cut.reshape([len(cut_percentages), \
                                       data_xe_r.shape[0], 2]).transpose(1,0,2)
all_y_hat_exp_cut = all_y_hat_exp_cut.reshape([len(cut_percentages), \
                                       data_xe_r.shape[0], 2]).transpose(1,0,2)

all_y_exp_cut = all_y_exp_cut.mean(axis=1)
all_y_hat_exp_cut = all_y_hat_exp_cut.mean(axis=1)

We plot a parity scatter of contact points obtained from mean-square-error fitting vs model predictions for the experimental 3T3 curves.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 5))

ax.scatter(all_y_exp_cut[:,0][labels==1], all_y_hat_exp_cut[:,0][labels==1], 
                alpha=0.4, 
                s=12)
lim = [all_y_exp_cut[:,0][labels==1].min(),
       all_y_exp_cut[:,0][labels==1].max()]
ax.plot(lim, lim, "k--", lw=1.5, label="Perfect")
ax.set_xlim(lim)
ax.set_ylim(lim)
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("MSE Fit Contact Point [um]", fontsize=15)
ax.set_ylabel("Predicted Contact Point [um]", fontsize=15)
ax.tick_params(axis='x', labelsize=20)
ax.tick_params(axis='y', labelsize=20)      
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(f'runs/{run_name}/results_cp_{run_name}.svg', bbox_inches="tight")
plt.savefig(f'runs/{run_name}/results_cp_{run_name}.png', bbox_inches="tight")
plt.tight_layout()
plt.show()

We plot a parity scatter of stiffness obtained from mean-square-error fitting vs model predictions for the experimental 3T3 curves.

In [ ]:
pre_y = (all_y_exp_cut[:,1]/1000)[labels==1]
pre_y_hat = (all_y_hat_exp_cut[:,1]/1000)[labels==1]

fig, ax = plt.subplots(1, 1, figsize=(12, 5))

ax.scatter(pre_y, pre_y_hat,
                color="#F0D43C",
                alpha=0.4,
                s=12)
ax.scatter(pre_y[pre_y<5], pre_y_hat[pre_y<5],
                color="#F0D43C",
                alpha=0.4,
                s=12)
lim = [0,5]
ax.plot(lim, lim, "k--", lw=1.5, label="Perfect")
ax.set_xlim(lim)
ax.set_ylim(lim)
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("True Stiffness [kPa]", fontsize=15)
ax.set_ylabel("Predicted Stiffness [kPa]", fontsize=15)
ax.tick_params(axis='x', labelsize=15)
ax.tick_params(axis='y', labelsize=15)      
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(f'runs/{run_name}/results_stiffness_{run_name}.png', \
            bbox_inches="tight")
plt.show()

Filters the input data, MSE fit, and model predictions to the subset of class-1 curves with stiffness below 5 kPa for the subsequent reconstruction quality analysis.

In [ ]:
all_x_rec = all_x_exp_cut[:data_xe_r.shape[0]][labels==1]
all_y_rec = all_y_exp_cut[labels==1]
all_y_hat_rec = all_y_hat_exp_cut[labels==1]

We loop over all selected curves and evaluates the Hertz contact model twice per curve; once with the ML-predicted parameters and once with the MSE fit parameters, storing the resulting force arrays for reconstruction loss comparison.

In [ ]:
approach_force_ml_list = []
approach_force_mse_list = []
approach_force_exp_list = []    

for indx in range(all_x_rec.shape[0]):
    
    approach_axis_um = all_x_rec[indx][0:1500]
    measure_force_pn = all_x_rec[indx][1500::]
    
    contact_point_um_mse = all_y_rec[indx][0]
    stiffness_pa_mse = all_y_rec[indx][1]
    
    contact_point_um_ml = all_y_hat_rec[indx][0]
    stiffness_pa_ml = all_y_hat_rec[indx][1]    

    
    approach_indent = np.maximum(
    contact_point_um_ml/1e6 - approach_axis_um/1e6, 0.0
    )
    approach_force_ml = contact_model.force(
        cantilever_shape=SimulationConfig.tip_shape,
        indentation_m=approach_indent,
        stiffness_pa=stiffness_pa_ml,
    )
    
    approach_force_ml_list.append(approach_force_ml*1e9)
    
    approach_indent = np.maximum(
    contact_point_um_mse/1e6 - approach_axis_um/1e6, 0.0
    )
    approach_force_mse = contact_model.force(
        cantilever_shape=SimulationConfig.tip_shape,
        indentation_m=approach_indent,
        stiffness_pa=stiffness_pa_mse,
    )

    approach_force_mse_list.append(approach_force_mse*1e9)
    approach_force_exp_list.append(measure_force_pn)

We stack the force lists into arrays and computes the per-curve mean-squared reconstruction error between the Hertz model output and the measured force signal, for both the ML-predicted and MSE fit parameter sets.

In [ ]:
approach_force_ml_list = np.array(approach_force_ml_list)
approach_force_mse_list = np.array(approach_force_mse_list)
approach_force_exp_list = np.array(approach_force_exp_list)
reconst_loss_ml = \
 ((approach_force_ml_list - all_x_rec[:,1500:])**2).mean(axis=1)
reconst_loss_mse = \
 ((approach_force_mse_list - all_x_rec[:,1500:])**2).mean(axis=1)

We create an interactive Plotly figure with a slider to browse all curves sorted by reconstruction loss. Each frame overlays the raw experimental signal, the ML Hertz fit (predicted parameters), and the reference Hertz fit (MSE fit parameters). Dashed vertical lines mark the ML contact point (red) and the MSE fit contact point (green).

In [ ]:
all_diff = reconst_loss_ml

# Sort by reconstruction loss
sorted_idx = np.argsort(all_diff)
y_list_sort = [approach_force_ml_list[i] for i in sorted_idx]
y_hat_list_sort = [approach_force_mse_list[i] for i in sorted_idx]
x_list_sort = [all_x_rec[i] for i in sorted_idx]
cp_ml_sort = [all_y_hat_rec[i][0] for i in sorted_idx]
cp_mse_sort = [all_y_rec[i][0] for i in sorted_idx]

diff_sort = [all_diff[i] for i in sorted_idx]

plotting.force_curve_prediction_widget(
    x_list_sort,
    y_list_sort,
    y_hat_list_sort,
    diff_sort,
    cp_ml_sort,
    cp_mse_sort,
)

We assemble the full run record and build a flat config dict, then instantiate `RunSaver` (which writes `config.json`). Training metrics (`metrics.csv`) and model checkpoints are already saved by Lightning's `CSVLogger` and `ModelCheckpoint` callbacks directly into `runs/<run_name>/`. Here we only parse the best val-loss from the CSV, compute contact-point and stiffness RMSE on the experimental 3T3 dataset, and save `results.json`.


In [ ]:
# Build config dict and instantiate the saver
# (run_name and model_structure were set in the model/training cell above)

# # full model
run_config = build_run_config(simulation_config, dataset_config, model_config)
saver      = RunSaver(
    run_name=run_name,
    config=run_config,
    model_structure=model_structure,
)

# Extract best val-loss and best epoch from the Lightning CSV
# (already saved by CSVLogger to runs/<run_name>/logs/version_0/metrics.csv)
metrics_csv = Path(csv_logger.log_dir) / "metrics.csv"
history = saver.load_history_from_csv(str(metrics_csv))
val_losses = history.get("val_loss", [])
if val_losses:
    saver.best_epoch    = int(np.argmin(val_losses)) + 1
    saver.best_val_loss = float(np.min(val_losses))

# Compute test metrics on the 3T3 real dataset (label == 1)
true_cp_um     = all_y_exp_cut[:, 0][labels == 1]
pred_cp_um     = all_y_hat_exp_cut[:, 0][labels == 1]
mask_stiff     = pre_y < 5
true_stiff_kpa = pre_y[mask_stiff]
pred_stiff_kpa = pre_y_hat[mask_stiff]

test_mse_cp         = float(mean_squared_error(true_cp_um, pred_cp_um))
test_rmse_cp        = float(np.sqrt(test_mse_cp))
test_mse_stiffness  = float(mean_squared_error(true_stiff_kpa, pred_stiff_kpa))
test_rmse_stiffness = float(np.sqrt(test_mse_stiffness))
num_params          = sum(p.numel() for p in best_regressor.parameters())

# Save results.json
# (config.json already saved by RunSaver.__init__;
#  metrics.csv and checkpoints already live in runs/<run_name>/ from Lightning)
results = saver.save_results(
    test_mse_cp=test_mse_cp,
    test_rmse_cp=test_rmse_cp,
    test_mse_stiffness=test_mse_stiffness,
    test_rmse_stiffness=test_rmse_stiffness,
    num_params=num_params,
)

print(json.dumps(results, indent=2))

We scan all `runs/*/results.json` files, merge them into a single ablation DataFrame, display it inline, and save it to `runs/ablation_table.csv` for easy comparison across runs.

In [ ]:
# Aggregate all saved runs into an ablation table
run_jsons = sorted(glob.glob("runs/*/results.json"))
rows = []
for p in run_jsons:
    with open(p) as f:
        rows.append(json.load(f))

if rows:
    ablation_df = pd.DataFrame(rows)
    display(ablation_df)
    ablation_df.to_csv("runs/ablation_table.csv", index=False)
    print(f"\nAblation table ({len(rows)} run(s)) saved to \
          runs/ablation_table.csv")
else:
    print("No runs found yet. Execute the saver cell first.")


### RMSE between experimental curves and Hertz fits

For each curve we compute the RMSE between the experimental force signal and the Hertz model evaluated at both the **MSE fit estimated** and the **predicted** parameters. Comparing the two distributions reveals whether the model's parameter estimates produce fits that are as faithful to the raw data as the MSE fit.

In [ ]:
rmse_ml = []
rmse_mse = []

for i in range(all_x_rec.shape[0]):
    rmse_ml.append(root_mean_squared_error(approach_force_exp_list[i], \
                                           approach_force_ml_list[i]))
    rmse_mse.append(root_mean_squared_error(approach_force_exp_list[i], \
                                           approach_force_mse_list[i]))

rmse_ml = np.array(rmse_ml)
rmse_mse = np.array(rmse_mse)

print(f"RMSE (mse params)  – mean: {rmse_mse.mean():.4f} nN  "
      f"median: {np.median(rmse_mse):.4f} nN")
print(f"RMSE (pred params)  – mean: {rmse_ml.mean():.4f} nN  "
      f"median: {np.median(rmse_ml):.4f} nN")
print(f"Curves where pred fit is better than truth: "
      f"{(rmse_ml < rmse_mse).sum()} / {all_x_rec.shape[0]}")

Plots overlapping RMSE histograms comparing reconstruction quality of the ML-predicted vs ground-truth Hertz fits, and a parity scatter (true RMSE vs predicted RMSE) to identify curves where the model's parameters yield a better fit than the reference.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overlapping histograms
axes[0].hist(
    rmse_mse,
    bins=40,
    alpha=0.55,
    color="#0188FFFA",
    label="True params"
    )
axes[0].hist(
    rmse_ml,
    bins=40,
    alpha=0.55,
    color="#AC982D",
    label="Pred params"
    )
axes[0].set_xlabel("RMSE [nN]", fontsize=15)
axes[0].set_ylabel("Count",     fontsize=15)
axes[0].set_title("RMSE distribution – good quality curves", fontsize=15)
axes[0].legend(fontsize=15)

# Scatter: RMSE_true vs RMSE_pred, diagonal = equal performance
max_lim = max(rmse_mse.max(), rmse_ml.max()) * 1.05
axes[1].scatter(rmse_mse, rmse_ml, color='black', alpha=0.2, s=14)
axes[1].plot([0, max_lim], [0, max_lim], "k--", lw=1.5, label="Equal RMSE")
axes[1].fill_between([0, max_lim], [0, 0], [0, max_lim],
                     alpha=0.1, color="#AC982D",
                     label="Pred better")
axes[1].fill_between([0, max_lim], [0, max_lim], [max_lim, max_lim],
                     alpha=0.1, color="#0188FFFA",
                     label="True better")
axes[1].set_xlim(0, max_lim)
axes[1].set_ylim(0, max_lim)
axes[1].set_aspect("equal", adjustable="box")
axes[1].set_xlabel("RMSE – true params [nN]",      fontsize=15)
axes[1].set_ylabel("RMSE – predicted params [nN]", fontsize=15)
axes[1].set_title("True vs Predicted RMSE",        fontsize=15)
axes[1].legend(fontsize=15)
for ax in axes:
    ax.tick_params(axis='x', labelsize=15)
    ax.tick_params(axis='y', labelsize=15)      
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(f'runs/{run_name}/results_rmse_{run_name}.png')
plt.show()